# N3 — PWM / Control Notebook
**Competencies** C6–C7, C13–C14 · **Artifacts** Duty-cycle characterization, Tuned Control Report · **Input** Family 3 CSV exports

Reproduce the duty curve and the control-response metrics from the exported traces; verify the thresholds.


## 1-2. Purpose & inputs
Inputs: `B7-duty-curve.csv`, `B8-step-response.csv`, `B9-coordinated-tracking.csv`, `B10-tuned-response.csv`.


In [1]:
from pathlib import Path
import csv, json, math, urllib.request
# Reference CSVs live in docs/figures. In-repo this resolves directly; on Colab the
# files are pulled from raw GitHub. Students can instead drop their own demo exports in DATA.
REPO_RAW='https://raw.githubusercontent.com/alibulentkoc/parallel-kinematics-hydraulics/main/docs/figures'
DATA = Path('../figures') if Path('../figures').exists() else Path('figures')
DATA.mkdir(exist_ok=True) if DATA==Path('figures') else None
def _ensure(name):
    p=DATA/name
    if not p.exists():
        try: urllib.request.urlretrieve(f'{REPO_RAW}/{name}', p); print('fetched', name)
        except Exception as e: raise FileNotFoundError(f'{name} not found locally and fetch failed: {e}')
    return p
def load_csv(name):
    rows=[]
    with open(_ensure(name)) as f:
        reader=csv.reader(l for l in f if not l.startswith('#'))
        header=next(reader)
        for r in reader:
            if r: rows.append(dict(zip(header,r)))
    return header, rows
def col(rows,k,cast=float):
    return [cast(r[k]) for r in rows if r.get(k) not in (None,'')]


## 3-4. Reproduce & analyze the duty curve (monotonic above deadband)


In [2]:
_,b7=load_csv('B7-duty-curve.csv')
duty=col(b7,'duty'); spd=col(b7,'avg_speed_mps')
mono=all(spd[i+1]>=spd[i]-1e-9 for i in range(len(spd)-1))
print('duty->speed monotonic:', mono)
assert mono, 'duty curve must be monotonic'


duty->speed monotonic: True


## On/off vs PWM vs proportional residual (from the step traces)


In [3]:
_,b8=load_csv('B8-step-response.csv')
def tail_err(key):
    xs=col(b8,key); tail=xs[int(len(xs)*0.8):]; return abs(0.10 - sum(tail)/len(tail))*1000
e_on=tail_err('x_onoff'); e_pwm=tail_err('x_pwm'); e_pr=tail_err('x_prop')
print(f'final error  on/off {e_on:.1f} mm | pwm {e_pwm:.1f} mm | prop {e_pr:.1f} mm  (on/off ~{e_on/e_pr:.0f}x prop)')


final error  on/off 11.2 mm | pwm 1.3 mm | prop 0.7 mm  (on/off ~17x prop)


## Coordinated tracking RMSE + tuned settling


In [4]:
_,b9=load_csv('B9-coordinated-tracking.csv')
sse=sum((float(r['target_x'])-float(r['actual_x']))**2+(float(r['target_y'])-float(r['actual_y']))**2 for r in b9)
rmse=math.sqrt(sse/len(b9))*1000
print(f'coordinated tracking RMSE {rmse:.1f} mm  (threshold <= 10 mm)')
_,b10=load_csv('B10-tuned-response.csv')
t=col(b10,'t'); x=col(b10,'x'); tol=0.10*0.02; settle=0.0
for ti,xi in zip(t,x):
    if abs(0.10-xi)>tol: settle=ti
print(f'tuned (PWM) settling {settle:.2f} s  (threshold <= 2.5 s)')


coordinated tracking RMSE 8.3 mm  (threshold <= 10 mm)
tuned (PWM) settling 0.47 s  (threshold <= 2.5 s)


## 5-6. Generate artifacts + verify


In [5]:
assert rmse<=10.0, 'tracking RMSE threshold'
assert settle<=2.5, 'settling threshold'
print('PASS: tracking <= 10 mm and settling <= 2.5 s')


PASS: tracking <= 10 mm and settling <= 2.5 s


## 7. Export


In [6]:
with open('duty-characterization.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['duty','avg_speed_mps']); w.writerows(zip(duty,spd))
with open('tuned-report.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['metric','value'])
    w.writerow(['tracking_rmse_mm',round(rmse,2)]); w.writerow(['settling_s',round(settle,2)])
    w.writerow(['onoff_final_err_mm',round(e_on,1)]); w.writerow(['prop_final_err_mm',round(e_pr,1)])
print('exported duty-characterization.csv, tuned-report.csv')


exported duty-characterization.csv, tuned-report.csv
